# 15j — multivariate scoring: WIS vs the energy & variogram score over age

9j scores the six forecast models with **WIS alone**, and WIS is *univariate*: it scores each
(origin × horizon × age bin) cell on its own and averages afterwards. So nothing in this project has
yet asked whether a model gets the **joint** forecast right — whether the 7 CIS age bins move
together as it says they should.

This notebook adds the two standard multivariate proper scoring rules for that question and
compares them against WIS **on identical forecasts**: it re-reads 9j's own assembly cache, so the
draws scored here are byte-for-byte the draws 9j scored.

| § | Question |
|---|---|
| §0 | Is the cache the one we think it is? |
| §1 | What does "ES over age" actually mean here — and why the horizon axis is not scored |
| §2 | What is in the draws, and which scale can these rules be read on? |
| §3 | The WIS baseline, at the full forecast unit |
| §4 | The multivariate grid, and the headline tables |
| §5 | Verification — two exactness tripwires and the R oracle |
| §6 | The age view: Σ WIS, ES over age, VS over age |
| §7 | Does the energy score rank the models the way WIS does — at every lead time? |
| §8 | How much dependence do these rules actually see? |
| §9 | Outputs |

**Three things to know before reading any panel below.**

1. **Everything is on the log scale**, the same `log(x + 1)` that `scoringutils` applies to WIS
   via `log_shift(offset = 1)` — §5 asserts the two are bit-identical. For these two rules that is
   not a reporting preference. WIS only ever sees the 5–95% quantiles, so it is structurally immune
   to the tail; ES and VS use **every** draw, and this grid contains extreme but perfectly *finite*
   draws (up to 3.8e109) that every existing guard passes. §2 measures what that does.
2. **"ES over X" names the axes that form the multivariate vector**, not a facet of a plot. §1
   spells it out.
3. **The only multivariate vector scored here is the 7-bin age vector at a fixed horizon.**
   Scoring a vector that spans the *horizon* axis was dropped on 2026-10-16, because the model as
   implemented emits **no cross-horizon joint predictive at all** — each horizon is a separate fit
   with its own RNG. Such a score is well defined, but it is a coupling this notebook would be
   imposing rather than one the forecast has. §1 states the mechanism and §8 reports the
   measurement that settled it.

⚠ **This notebook cannot fit anything.** Every read is `isfile` → `load` of one named `.jld2`, and
nothing here calls `two_stage_forecast` / `fit_or_load_*` / `assemble_or_load_forecasts`. §5 asserts
that mechanically rather than trusting this paragraph.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
include("9j_viz_utils.jl")     # REF_MODEL, WIS_SCALE, panel_grid, save_show, PERIODS,
                               #   shade_periods!, period_of; it includes 8j_viz_utils.jl too
include("15j_viz_utils.jl")    # this notebook's helpers
using Random, Statistics, StatsBase, DataFrames, Printf, CSV
mkpath("../res")

default_plot_setting()

In [ ]:
# 15j is a READ-ONLY notebook, so it follows 14j's pattern rather than 9j's: it takes the
# framework default `stage1_use_nuts = true` and ASSERTS it, instead of reading
# `ENV["STAGE1_USE_NUTS"]`. Reading that variable would only create a way to point the notebook at
# a generation that was never fitted — which for 9j/10j means a silent, days-long refit. 15j cannot
# refit at all, but it CAN point at a cache that does not exist, and an assert up front is a much
# better failure than one 300 lines down.
cfg  = FrameworkConfig(constant_contacts = false)   # production defaults: per-week contacts, NUTS
grid = cis_age_grid()

@assert cfg.stage1_use_nuts
@assert contacts_label(cfg) == "temporal-w8h-lc0-eng-nuts"
@assert WIS_SCALE == MV_SCALE == "log"   # WIS, ES and VS must be on ONE scale (see §2)

# The SIX model variants, in 9j's fixed order — built here independently of the cache so that the
# comparison `load_9j_assembly` makes is a real cross-check rather than a tautology.
combos = vcat([(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                        for nb in (MeanNGM(), NeighbourhoodDegreeNGM())],
              [(NegBinAgePair(),   DiagonalMeanNGM()),      # no-interaction  (= REF_MODEL)
               (NoContactDegree(), NullNGM())])             # null
LABELS = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]

@printf("token      %s\ngrid       %s\nmodels     %d\nref model  %s\n",
        contacts_label(cfg), join(grid.LAB, ", "), length(LABELS), REF_MODEL)
@assert REF_MODEL in LABELS

## §0 — The cache: is it the one we think it is?

`load_9j_assembly` is `isfile` → `load` → check, and nothing else. It deliberately does **not** call
`assemble_or_load_forecasts`, which needs the CoMix `raw` read and, on a miss, falls through to
`two_stage_forecast` → `fit_or_load_stage2` → `fit_or_load_stage1` and *fits*. Keeping that function
out of 15j's call graph entirely is the no-refit guarantee.

It validates the model labels against the list built above, the origins' shape (length, endpoints,
sortedness, exact weekly spacing), that all 378 `(origin, model)` fans are present, and that each
fan is `7 × 4 × 10 000`.

⚠ **It also checks something the cache itself cannot.** 9j's cache validates `origins` and `labels`
— *names, not contents*. On 2026-09-16 the infection source moved from `estimates_age_ab.csv` to
`estimates_age.csv`; origins and labels were both unchanged, so a stale cache would have been served
silently and had to be deleted by hand. So the loader compares the cache's mtime against the
estimates CSV and the newest `8j_s2_*` artefact, and warns if either is newer. A docstring warning
nobody runs is worth less than two `mtime` comparisons.

The truth comes from the cache too: `to_quantile_long` writes the observation into every row as
`observed`, so the values 9j actually scored against are already in `qall` and **no inc2prev read is
needed**. `assembly_truth` asserts the observation is *model-invariant* — one distinct value per
(origin, horizon, age bin) — which simultaneously proves every model was scored against the same
truth and that the unit sets coincide. That is the precondition for comparing any two rules at all.

In [ ]:
asm = load_9j_assembly(cfg; labels = LABELS, grid = grid)
tru = assembly_truth(asm.qall, grid; horizons = cfg.horizons)

@printf("cache      %s\n           %.0f MB, written %s\n",
        asm.path, asm.bytes / 1e6, Dates.unix2datetime(asm.mtime))
@printf("origins    %d weekly, %s … %s\n", length(asm.origins), first(asm.origins),
        last(asm.origins))
@printf("fans       %d  (= %d origins × %d models), each %s\n", length(asm.fc_store),
        length(asm.origins), length(asm.labels),
        string(size(asm.fc_store[(first(asm.origins), first(asm.labels))])))
@printf("quantiles  %d rows   truth cells %d\n", nrow(asm.qall), tru.n_cells)

## §1 — What "ES over age" means here, and why the horizon axis is not scored

The **energy score** is the multivariate generalisation of CRPS. For a `d`-vector observation `y`
and `M` draws `x_m`:

$$\mathrm{ES} = \frac1M\sum_m \lVert x_m - y\rVert \;-\; \frac{1}{2M^2}\sum_i\sum_j \lVert x_i - x_j\rVert$$

The **variogram score** of order `p` compares the sample's pairwise structure to the observation's:

$$\mathrm{VS}_p = \sum_i\sum_j\Bigl(\,\lvert y_i - y_j\rvert^p - \tfrac1M\textstyle\sum_m \lvert x_{mi} - x_{mj}\rvert^p\Bigr)^2$$

**"over X" names the axes that form the vector**, which is what decides how many scores there are
and what each one is about. The cell below prints the reported choices from the code itself.

### Why only `age`

⚠ **Across the horizon axis, the model produces no joint predictive at all.** In
`two_stage_forecast` each horizon is a *separate* `fit_or_load_stage2` with its own pooled draws and
its own `MersenneTwister(cfg.seed + h)`, and the intervening-horizon lag plug is the **median** over
finite draws — a point summary, so no draw-level path is shared. The function's own docstring says
"per-draw coherence across horizons is undefined".

An "ES over time" is therefore a **definition** — score the independence coupling of the four
marginal forecasts — and not an estimate of anything the model emits. It is a perfectly valid proper
score of a perfectly well-defined predictive distribution, and all six models are coupled the same
way, so the comparison would be fair. It is dropped anyway, for one reason: a reader of a figure
labelled "energy score over time" will take it to say something about temporal dependence, and it
cannot. §8 carries the measurement that makes this concrete — destroying the cross-horizon draw
pairing entirely moved ES by ≤0.06% and VS by ≤0.8%, because there was no coupling there to destroy.

So since **2026-10-16** the reported margins are just two:

- **`age`** — the 7-bin CIS vector at one horizon. The one genuinely multivariate view.
- **`none`** — `d = 1`, the univariate baseline. Not a view: it is where ES degenerates exactly to
  `crps_sample` (§5's sharpest assert) and the only place WIS and CRPS meet on an identical cell.

Both horizon-spanning margins remain *computable* — pass `margins = ("time", "age-time")` — they
are simply not reported.

### Two properties to read off the figures, not just note here

- **ES at `d = 1` is exactly `crps_sample`.** Not an approximation — the `1/M²` convention in
  `energy_score` is the same one `crps_sample` and `scoringRules::es_sample` use. §5 asserts it over
  10 584 real groups.
- **VS is shift-invariant and is not proper for the marginals.** It sees only pairwise
  *differences*, so a model biased uniformly high is not penalised by it at all. It therefore
  *complements* the energy score and must never be read as a substitute — §7 shows what that costs
  in practice. We report **`p = 0.5`** as the headline (named in every column and title) and compute
  `p = 1` alongside only to show whether the ranking is `p`-stable — free, because VS is `O(M·d²)`
  against ES's `O(M²·d)`.

Across the **age** axis there *is* a genuine shared-parameter joint at fixed (horizon, draw): one
contact matrix, one susceptibility/infectivity/γ_SAR/generation-interval draw, and the matrix
product `N · acc` couples all 7 bins. But the observation layer `p + σ·randn` is **iid across age**,
and §8 shows it dominates.

In [ ]:
# The reported margins, straight from `_margin_groups` so this table cannot drift from the code.
# `MARGIN_ORDER` is the single source of truth for which ones are REPORTED; the dropped two are
# still dispatchable, which is why the labels below cover all four.
rows = NamedTuple[]
for m in MARGIN_ORDER
    g = _margin_groups(m, grid.N, length(cfg.horizons))
    push!(rows, (margin = m, d = length(first(g).idx), groups_per_fan = length(g),
                 groups_per_model = length(g) * length(asm.origins),
                 free_keys = m == "none"     ? "origin × horizon × age" :
                             m == "time"     ? "origin × age" :
                             m == "age"      ? "origin × horizon" : "origin",
                 vector_spans = m == "none"  ? "— (univariate baseline)" :
                                m == "time"  ? "the 4 horizons" :
                                m == "age"   ? "the 7 CIS age bins" : "all 28 age × horizon cells"))
end
show(DataFrame(rows); allrows = true, allcols = true, truncate = 40); println()

# State the drop where the table is read, not only in the prose above it.
dropped = [m for m in ("none", "time", "age", "age-time") if m ∉ MARGIN_ORDER]
println("\nNOT reported: ", isempty(dropped) ? "(none)" : join(dropped, ", "),
        "  — the model emits no cross-horizon joint predictive, so a vector spanning the horizon")
println("axis is a coupling this notebook would impose rather than one the forecast has (§1, §8).")
@assert Set(MARGIN_ORDER) == Set(("none", "age")) "15j: §1's prose describes exactly the margins " *
    "none + age; MARGIN_ORDER says $(MARGIN_ORDER) — update the markdown or the constant."

## §2 — What is in the draws, and which scale these rules can be read on

`mv_scores` computes ES and VS on **both** scales here, once, because the natural-scale numbers are
the *evidence* for why everything downstream is log-scale. They are never plotted and never quoted
as a headline.

The reason is a guard class that `score_wis` cannot see. `score_wis` drops whole forecast units for
**non-finite** predictions, and there are none in this grid. But the fans contain extreme
**finite** draws — a pathological Pathfinder draw gives a supercritical NGM, which
`two_stage_forecast` deliberately keeps rather than fabricating a NaN. WIS never notices, because
`to_quantile_long` keeps only the 5–95% quantiles and throws the tail away. ES and VS are means over
*every* draw, so one such draw is the whole score.

`M` is also **equalised across models** before scoring: ES's sample estimator is biased and its bias
depends on `M`, so two groups scored at different `M` are two different estimators. ⚠ That is *not*
aligning draws — draw index `d` means nothing across models, since each has its own Stage-1/Stage-2
fits and its own observation RNG. The cut only equalises the estimator's bias.

In [ ]:
SCALES = ("log", "natural")   # natural is computed ONCE, for the audit below only
mv = mv_scores(asm.fc_store, tru.truth, asm.labels, asm.origins;
               grid = grid, cfg = cfg, scales = SCALES)
@printf("\nmv_scores: %d rows (%d groups per model per scale)\n", nrow(mv.scores),
        nrow(@subset(mv.scores, :model .== first(asm.labels), :scale .== "log")))

println("\n===== draw audit =====")
show(mv.audit; allrows = true, allcols = true, truncate = 34); println()

# All six models must be scored on identical units and an identical M, or part of the comparison is
# a comparison of M. Printed as an all-clear rather than hidden in a passing assert.
@assert length(unique(mv.scores.M)) == 1
@assert sum(mv.scores.n_draw_dropped) == 0
let g = combine(groupby(@subset(mv.scores, :scale .== "log"), :model), nrow => :n)
    @assert length(unique(g.n)) == 1 "models do not have identical group sets"
    @printf("\ncommon units: all %d models on %d groups each, M = %d, 0 draws dropped\n",
            nrow(g), first(g.n), first(mv.scores.M))
end

In [ ]:
# THE finding: what the natural scale does to a multivariate score, and what log fixes.
ev = combine(groupby(@subset(mv.scores, :margin .== "age"), [:scale, :model]),
             :es => median => :es_median, :es => mean => :es_mean, :es => maximum => :es_max,
             :vs_p05 => median => :vs_median, :vs_p05 => maximum => :vs_max,
             :n_extreme => sum => :n_extreme, :max_abs_draw => maximum => :max_abs_draw)
println("ES / VS(p=0.5) over the 7-bin age vector, by scale — note es_mean vs es_median:")
show(sort(ev, [:scale, :model]); allrows = true, allcols = true, truncate = 32); println()

@printf("\nworst single draw anywhere on the grid: %.3g\n", maximum(mv.scores.max_abs_draw))
@printf("draws above |%.0e|: %d  (finite, so `score_wis` passes every one of them)\n",
        EXTREME_DRAW, sum(@subset(mv.scores, :margin .== "none", :scale .== "log").n_extreme))
@printf("ratio mean/median of the natural-scale ES, worst model: %.3g\n",
        maximum(r -> r.es_mean / r.es_median, eachrow(@subset(ev, :scale .== "natural"))))
@printf("    ... and on the log scale:                          %.3g\n",
        maximum(r -> r.es_mean / r.es_median, eachrow(@subset(ev, :scale .== "log"))))

# From here on, ONLY the log scale.
MV = @subset(mv.scores, :scale .== MV_SCALE)
@printf("\nproceeding on the %s scale: %d rows\n", MV_SCALE, nrow(MV))

## §3 — The WIS baseline, at the full forecast unit

WIS is computed from the cache's own `qall` — **not** from `res/8j_scores_*.csv`. Two reasons: those
CSVs lack the finest grain, and reading a CSV written by a different run would reintroduce exactly
the names-not-contents hazard §0 guards against. Scoring the same file that holds the draws makes
"WIS and ES come from the same forecasts" provable rather than assumed.

`score_wis` gained a sixth frame, `by_model_dt_h_age`, for this notebook. The matched comparator for
"ES over the 7 age bins at (model, origin, horizon)" is the **sum** of WIS over those 7 bins at that
origin and horizon — and the pre-existing `by_model_h_age` has already averaged over origins before
you can take that sum. A metric comparison is only honest if the aggregation is held fixed;
otherwise part of what you measure is the aggregation choice. `wis_aggregate` does that summing, per
margin, onto the same keys `mv_scores` uses.

In [ ]:
scores = score_wis(asm.qall)
wl     = wis_long(scores; scale = WIS_SCALE)
wisagg = wis_aggregate_all(wl)
joined = mv_join_wis(MV, wisagg; scale = MV_SCALE)

@printf("by_model_dt_h_age : %d rows (both scales)\n", nrow(scores.by_model_dt_h_age))
@printf("matched WIS       : %d rows over %d margins\n", nrow(wisagg),
        length(unique(wisagg.margin)))
@printf("joined            : %d rows, %d without a matched WIS\n", nrow(joined),
        count(ismissing, joined.wis_sum))
@assert count(ismissing, joined.wis_sum) == 0

# The two audits measure the SAME forecasts through two different pipelines — WIS through the
# quantiles, ES through the draws — so they should agree about which models misbehave. If they
# disagree, one of the two is reading something wrong.
println("\nscore_wis's own quantile-side audit:")
@printf("  negative predicted quantiles : %d\n", scores.n_negative)
@printf("  non-finite quantiles         : %d\n", scores.n_nonfinite)
nrow(scores.negative_by_model) > 0 && show(scores.negative_by_model; allrows = true)
println()

## §4 — The multivariate grid and the headline tables

Relative skill is the **median of the per-group ratios** to `REF_MODEL`, with the mean reported
beside it. Even on the log scale a mean is outlier-sensitive — 9j's own `crps` frame has one cell at
5.14e7 against ~5.5e4 for its peers — so when the median and the mean disagree about a winner,
**that disagreement is the result**: it says one origin is carrying the mean.

ES and VS are both non-negative, so unlike the log score that was removed from this project in
2026-08 a *ratio* is interpretable here and no difference-based workaround is needed.

In [ ]:
summ = summarise_mvscores(joined; ref = REF_MODEL)
conc = score_concordance(joined, summ)

for mg in MARGIN_ORDER
    s = @subset(summ, :margin .== mg)
    nrow(s) == 0 && continue
    println("\n===== margin = ", mg, " =====")
    show(select(sort(s, [:metric, :rank_median]),
                [:metric_label, :model, :n_groups, :score_median, :score_mean,
                 :rel_median, :rel_mean, :rank_median]);
         allrows = true, allcols = true, truncate = 34)
    println()
end

## §5 — Verification

Six checks, in rough order of how much they would catch.

1. **ES reduces to `crps_sample` at `d = 1`, exactly** — asserted over every `margin = "none"` row
   on the grid. One assert covers the primitive, the `1/M²` convention, the `d × M` orientation and
   the truth lookup at once.
2. **The cache's own `crps` frame is reproduced** — `mean_crps` was computed by a *different run of
   a different function* against truth loaded from inc2prev, so reproducing all 378 of its values
   proves that the truth recovered from `qall` is the truth 9j scored against, and that the fan's
   `(age, horizon)` orientation matches `grid.LAB`. **A transposed index passes every shape assert
   and fails this one.**
3. **The log transform is WIS's log transform.** If `log_shift`'s offset or base convention ever
   changed, ES/VS and WIS would drift onto different scales while both were still called "log", and
   nothing downstream would notice. ⚠ The claim is agreement to a few **ulp**, not bit-identity:
   `scoringutils` and Julia call different libm `log` implementations, and on 20 000 real values a
   handful disagree in the last bit. The tolerance is still four orders of magnitude tighter than
   any convention error could hide in — a wrong offset or base moves the result by O(1).
4. **The native scores match `scoringRules`** on real groups. This is the only thing that pins the
   variogram normalisation — the full ordered double sum versus the `i<j` half is a factor of 2.
   ⚠ `scoringRules` is installed here but is **not** declared in the devcontainer, so the check is
   guarded and skips rather than fails; the native implementation is the engine either way.
5. **`scoringutils`' `joint_across` means what this notebook assumes** — "which margins form the
   vector" — checked on one small group through the real 2.2.0 multivariate API.
6. **Nothing in the helper file can fit**, checked by grep rather than by assertion in a comment.

In [ ]:
# 1 — ES == crps_sample at d = 1, through the GENERAL O(M²) kernel.
# ⚠ This must NOT be read off the `margin = "none"` rows: `mv_scores(…; fast_univariate = true)`
# fills those from `crps_sample` directly, so comparing them would be circular. The identity is
# what licenses that shortcut, so it is verified here on real cells through the slow path.
RED = verify_es_reduces_to_crps(asm.fc_store, tru.truth, asm.labels, asm.origins;
                                grid = grid, cfg = cfg, n = 400, scale = MV_SCALE, rtol = 1e-10)
@printf("1. log scale, 400 real cells : max rel = %.3e\n", RED.max_rel)

# The same check on the NATURAL scale, at a deliberately looser tolerance — and the looseness is
# the point. The pair term sums 5e7 square roots, and in a cell holding a 3.8e109 draw beside
# ~1e5 ones Float64 cannot represent the small contributions at all, so this O(M²) kernel and
# `crps_sample`'s sorted O(M log M) rearrangement necessarily lose different amounts. The identity
# is algebraically exact; it is the natural scale that cannot carry it. One more reason §2's
# answer is "log".
REDN = verify_es_reduces_to_crps(asm.fc_store, tru.truth, asm.labels, asm.origins;
                                 grid = grid, cfg = cfg, n = 400, scale = "natural", rtol = 1e-6)
@printf("   natural scale, same cells  : max rel = %.3e  (%.0f× worse, and irreducibly so)\n",
        REDN.max_rel, REDN.max_rel / RED.max_rel)

# 2 — reproduce 9j's own `crps` frame (natural scale, all 28 cells)
nat  = @subset(mv.scores, :margin .== "none", :scale .== "natural")
mine = combine(groupby(nat, [:model, :origin]), :crps_mean => mean => :mean_crps)
jc   = innerjoin(mine, rename(asm.crps, :mean_crps => :ref_crps); on = [:model, :origin])
rel2 = abs.(jc.mean_crps .- jc.ref_crps) ./ abs.(jc.ref_crps)
@printf("2. reproduces 9j's `crps` over %d (model,origin) : max rel = %.3e\n",
        nrow(jc), maximum(rel2))
@assert nrow(jc) == length(asm.origins) * length(asm.labels)
@assert maximum(rel2) < 1e-10

# 3 — the log scale IS WIS's log scale
LOGCHK = log_scale_matches_wis(asm.qall)

# 4 / 5 — the guarded R oracle
XCHK = es_cross_check_r(asm.fc_store, tru.truth, asm.labels, asm.origins; grid = grid, cfg = cfg)
XCHK === nothing || show(XCHK; allrows = true, allcols = true, truncate = 26); println()
SEM  = scoringutils_semantics_check(asm.fc_store, tru.truth, asm.labels, asm.origins;
                                    grid = grid, cfg = cfg)

# 6 — 15j cannot fit. A claim in a header comment is not a check, so grep for it.
# Two refinements that matter, both learned by getting them wrong first: the docstrings in
# `15j_viz_utils.jl` NAME these functions (that is how they explain what 15j does not do), so the
# triple-quoted blocks have to be stripped as well as the `#` comments; and the pattern must
# require CALL syntax (`name(`), because a mention is not a call.
let src = read("15j_viz_utils.jl", String)
    nodoc = replace(src, r"(?s)\"\"\".*?\"\"\"" => "")
    code_lines = [l for l in split(nodoc, "\n") if !startswith(strip(l), "#")]
    pat = r"\b(two_stage_forecast|fit_or_load_\w*|prefit_\w*|assemble_or_load_forecasts|load_forecast_truth|load_window_data|prepare_degree_data)\s*\("
    hits = [l for l in code_lines if occursin(pat, l)]
    @assert isempty(hits) "15j_viz_utils.jl contains a FITTING CALL:\n" * join(hits, "\n")
    # NB `@printf` needs a LITERAL format string — `"a" * "b"` is an expression and the macro
    # rejects it ("First argument to `@printf` after `io` must be a format string").
    println("6. no fitting call in 15j_viz_utils.jl (", length(code_lines),
            " code lines, after stripping ", length(src) - length(nodoc),
            " chars of docstring and the comments)")
end

## §6 — The age view

ES over the 7-bin CIS age vector, one score per (model, origin, horizon), against the **matched**
Σ WIS — the full-unit WIS summed over exactly the same 7 cells — and the variogram score at
**p = 0.5**. Same models, same colours, same axes: three panels side by side *is* the comparison.

This used to be three figures, one per margin. The other two scored vectors spanning the horizon
axis and were dropped on 2026-10-16 (§1); what replaces them as the "over time" view is §7's
cumulative difference against a fixed reference, which is a statement about forecast *origins* and
not about a joint the model never produced.

Colours match 9j's, so a model carries the same colour here as in `res/8j_*` and `res/9j_*`.

In [ ]:
save_show(plot_rel_score_by_horizon(joined, asm.labels, cfg),
          "../res/15j_rel_score_by_horizon.png");

### Is the log scale enough?

A `d`-dimensional energy score is a Euclidean norm, so if one coordinate is much larger than the
others the score is effectively one-dimensional. The check is the share of the squared error each
coordinate carries. **Decision rule: if every share sits within ~2× of uniform the log scale
suffices; if one coordinate carries more than ~40% the vector is effectively collapsed and a
truth-derived per-coordinate standardisation would be needed.**

This is why no population normalisation is applied: dividing by `grid.POP` would change the question
from "who forecasts the epidemic" to "who forecasts incidence rates", and nothing else in this
project is per-capita.

In [ ]:
shares = coordinate_shares(asm.fc_store, tru.truth, asm.labels, asm.origins;
                           grid = grid, cfg = cfg)
show(shares; allrows = true, allcols = true); println()

worst = maximum(shares.share)
@printf("\nlargest coordinate share %.1f%%; uniform would be %.1f%% (age) / %.1f%% (horizon)\n",
        100 * worst, 100 / grid.N, 100 / length(cfg.horizons))
if worst > 0.40
    @warn "a single coordinate carries >40% of the squared error — the log scale alone is NOT " *
          "enough and a truth-derived per-coordinate standardisation is needed before these " *
          "multivariate scores can be read as scores of the whole vector"
else
    println("=> within the decision rule: the log scale is sufficient, no standardisation applied")
end
save_show(plot_coordinate_shares(shares), "../res/15j_coordinate_shares.png");

## §7 — Does the energy score rank the models the way WIS does?

This is the question the whole notebook exists to answer, and it is asked four ways:

- a **bump chart** of each model's rank under each rule — rank being the only honest common scale
  across rules whose units differ, which is why this is not a dual-axis comparison of raw scores;
- the same bump chart **split by lead time**, which is where the interesting disagreement lives;
- a **per-group scatter** of Σ-WIS against ES, faceted by model, with Spearman ρ — the
  better-powered question of whether the two rules agree about which *origins* a model got wrong;
- **ES against Σ CRPS**, to see how much of the energy score is just the marginals.

⚠ The last one is a **rank check, not an equality check**. ES is a *norm* and Σ CRPS is a *sum*: for
`d` coordinates of comparable scale ES grows like `√d·scale` while Σ CRPS grows like `d·scale`, so
the ratio is `O(1/√d)` and *should* sit well below 1 for a correct implementation. Equality holds
only at `d = 1`, which is §5's first assert.

### Two things you need in order to read these panels correctly

**1. The MEAN ranking cannot move with the aggregation. The MEDIAN ranking can.** The mean over
groups of a sum over collapsed cells is the total divided by the number of groups, and that total is
the same cells however they are grouped — so a mean ranking is invariant to the margin and to the
horizon split. A median of sums is not a sum of medians, so a median ranking is not. Measured on
this grid for `weighted-hweibull|mean` vs `unweighted-negbin|mean`, the two models nearest to tied:

| | median/cell | mean/cell | mean ÷ median |
|---|---|---|---|
| hweibull\|mean — per cell (`none`) | **0.3087** | 0.4683 | 1.517 |
| negbin\|mean — per cell | 0.3334 | 0.5109 | 1.532 |
| hweibull\|mean — summed over 28 cells | 0.3993 | 0.4683 | 1.173 |
| negbin\|mean — summed over 28 cells | **0.3980** | 0.5109 | 1.284 |

The mean per cell is *literally unchanged* by the aggregation (0.4683, 0.5109) — that is the
additivity, directly. Summing cells averages away each model's per-cell right skew at *different*
rates (1.517→1.173 vs 1.532→1.284), the median climbs toward the mean, and two series 7% apart that
move at different rates cross. **A rank swap between panels is a property of the summary statistic,
not a disagreement between scoring rules** — which is exactly why the per-horizon figure below
draws the median and the mean as two rows of the same chart.

**2. What `margin = "none"` compares.** The group is a single (origin, horizon, age) cell, so
`es == crps_sum == crps_mean` to `0.000e+00` and `vs_p05` is NaN on all 10 584 rows — a single
coordinate has no pairs. The only live comparison on that panel is **WIS against CRPS on the
identical cell**: 19 quantile levels against all 10 000 draws. Measured WIS/CRPS median ratio
**1.0452** (median relative difference 4.52%, p90 5.18%, max 5.6%) — the expected discretisation gap
from approximating CRPS on a finite quantile grid, near-uniform across models, which is why the two
rank identically there. That panel is the baseline the `age` panel is read against.

In [ ]:
save_show(plot_score_rank_bump(summ, asm.labels), "../res/15j_score_rank_bump.png");

println("\nrank agreement between the rules (6 model medians, 15 model pairs):")
show(conc.model_level; allrows = true, allcols = true, truncate = 28); println()
println("\np-stability of the variogram ranking (p=0.5 vs p=1):")
show(@subset(conc.model_level, :metric_x .== "vs_p05", :metric_y .== "vs_p1");
     allrows = true, allcols = true, truncate = 28); println()

In [ ]:
# The SAME ranking, split by lead time. `margin = "age"` is the only reported multivariate margin
# with a free horizon axis, so this is where "does the ranking hold?" can actually be asked.
#
# Two rows on purpose: top = rank by the MEDIAN across the 63 origins, bottom = by the MEAN. §7's
# markdown has the arithmetic; the picture is that the mean ranking of an ADDITIVE rule cannot move
# with the aggregation and the median ranking can, so printing one without the other invites the
# reader to attribute a summary-statistic artefact to the scoring rule.
summ_h = summarise_mvscores_by_horizon(joined; ref = REF_MODEL)

save_show(plot_score_rank_bump_by_horizon(summ_h, asm.labels),
          "../res/15j_score_rank_bump_by_horizon.png");

# The headline contrast, as a sentence rather than a figure to be squinted at.
println("\nis the model ordering IDENTICAL at every horizon (margin = age)?")
for met in ("wis_sum", "crps_sum", "es", "vs_p05", "vs_p1")
    sm = @subset(summ_h, :metric .== met)
    nrow(sm) == 0 && continue
    ordm = Set(tuple(sort(@subset(sm, :horizon .== h), :rank_median).model...)
               for h in sort(unique(sm.horizon)))
    orda = Set(tuple(sort(@subset(sm, :horizon .== h), :rank_mean).model...)
               for h in sort(unique(sm.horizon)))
    println("   ", rpad(mv_metric_label(Symbol(met)), 17),
            " by median: ", rpad(length(ordm) == 1 ? "YES" : "NO ($(length(ordm)) orderings)", 18),
            " by mean: ", length(orda) == 1 ? "YES" : "NO ($(length(orda)) orderings)")
end

# ⚠ The expected result, stated so a change is noticed: Σ WIS, Σ CRPS and ES — the three PROPER,
# marginal-sensitive rules — give ONE ordering across all four lead times on the mean, and both
# variogram orders do not. That contrast is the cleanest evidence here that VS measures a different
# quantity. Its sharpest form: at h = 4 by mean, VS(p=1) ranks `no-contact|null` FIRST — a model
# with a constant C* and no contact data at all. That is VS's impropriety for the marginals, not a
# finding about the null model.
for met in ("wis_sum", "crps_sum", "es")
    sm = @subset(summ_h, :metric .== met)
    o = Set(tuple(sort(@subset(sm, :horizon .== h), :rank_mean).model...)
            for h in sort(unique(sm.horizon)))
    length(o) == 1 || @warn "15j: `$(met)` no longer gives one MEAN ordering across horizons — " *
                            "§7's markdown and the figure caption both assert that it does."
end
println("\nbest model by MEAN at each horizon (Σ WIS / Σ CRPS / ES agree):")
for h in sort(unique(summ_h.horizon))
    b = first(sort(@subset(summ_h, :horizon .== h, :metric .== "es"), :rank_mean))
    v = first(sort(@subset(summ_h, :horizon .== h, :metric .== "vs_p1"), :rank_mean))
    println("   h=", h, "  ES: ", rpad(b.model, 34), "  VS(p=1): ", v.model)
end

In [ ]:
save_show(plot_score_vs_wis(joined, asm.labels; margin = "age"),
          "../res/15j_score_vs_wis.png");

println("\nper-model Spearman ρ between Σ WIS and ES, by margin:")
show(unstack(@subset(conc.group_level, :metric_x .== "wis_sum", :metric_y .== "es"),
             :model, :margin, :spearman); allcols = true, truncate = 22); println()

In [ ]:
save_show(plot_es_vs_sumcrps(joined, asm.labels), "../res/15j_es_vs_sumcrps.png");

### Cumulative skill difference over time

9j's `plot_wis_diff_over_time` (`res/9j_wis_diff_over_time.png`) accumulates each model's
per-origin WIS gap against a fixed reference in date order, so a line's endpoint is the
whole-period difference and its slope is where the difference was earned. These are the same figure
for each of the four scores, with the **same reference** 9j uses
(`weighted-hweibull|neighbourhood`) and the same two drawn models **plus `no-contact|null`**, so the
panels read directly against 9j's.

The reference is omitted (identically 0, the dashed line). Below 0 beats it cumulatively; a steadily
rising line loses a little every week.

⚠ **Σ WIS here is a SUM over the 7 age bins; 9j's `by_model_dt_h` WIS is a MEAN over them.** A
constant factor of 7 — the shapes are identical and comparable with 9j's figure, the y-axis
magnitudes are not.

⚠ **For the two variogram panels this is a weaker statement than for Σ WIS or ES.** VS is
shift-invariant and not proper for the marginals, so a uniformly biased forecast is not penalised at
all. A rising VS line means the model's pairwise *dependence* structure is drifting from the truth's
— not that it forecasts worse. The bump chart above shows what happens if you forget that.

In [ ]:
# One figure per score, each faceted by horizon — 9j's layout exactly.
REF_DIFF    = "weighted-hweibull|neighbourhood"          # 9j's reference, kept so the two compare
MODELS_DIFF = ["unweighted-negbin|mean", "weighted-hweibull|mean", "no-contact|null"]
@assert REF_DIFF ∈ asm.labels && all(m -> m ∈ asm.labels, MODELS_DIFF)

DIFF_METRICS = ((:wis_sum, "wis"), (:es, "es"), (:vs_p05, "vs_p05"), (:vs_p1, "vs_p1"))
diff_frames = DataFrame[]
for (met, stem) in DIFF_METRICS
    push!(diff_frames, score_diff_frame(joined; metric = met, ref = REF_DIFF, models = MODELS_DIFF))
    save_show(plot_score_diff_over_time(joined, asm.labels, cfg;
                                        metric = met, ref = REF_DIFF, models = MODELS_DIFF),
              "../res/15j_score_diff_over_time_$(stem).png");
end
diffs = reduce(vcat, diff_frames)

# Endpoints = the whole-period difference vs the reference. This is the table the figures are a
# picture of, and it is the only place the sign of each line is stated in words.
ends = combine(groupby(diffs, [:metric, :horizon, :model]), :cum_diff => last => :whole_period)
println("\nwhole-period cumulative difference vs ", REF_DIFF, " (< 0 ⇒ beats it):")
for (met, _) in DIFF_METRICS
    println("\n--- ", mv_metric_label(met), " ---")
    show(unstack(@subset(ends, :metric .== String(met)), :model, :horizon, :whole_period);
         allrows = true, allcols = true, truncate = 34)
    println()
end
@assert nrow(diffs) == length(DIFF_METRICS) * length(MODELS_DIFF) *
                       length(cfg.horizons) * length(asm.origins) """
    15j: the diff frame should be one row per (metric, model, horizon, origin); got $(nrow(diffs))"""

## §8 — How much dependence do these rules actually see?

Permute **each coordinate's draws independently**. That preserves every marginal exactly — asserted
inside the function, to 0.0, by comparing sorted rows — and destroys all dependence between
coordinates. Then re-score. The relative change is how much of the score was coming from the joint
structure rather than from the marginals.

The test now runs on the **age** margin only, which is the only vector the model produces jointly.
There *is* a shared-parameter joint across age, so a non-zero answer is possible, and what it
measures is how much of that joint survives the iid-across-age observation layer.

⚠ **This section is also the evidence behind §1's drop.** Run on the horizon-spanning margins it
returned **ES ≤0.06% and VS ≤0.8%** — destroying the cross-horizon draw pairing entirely barely
moved the score, because the model never put any coupling there to destroy. That is a structural
expectation, not a null that was tested and failed to reject. Pass `margins = ("time",)` to
reproduce it; do not read it as a finding about the data.

⚠ No tolerance is asserted on the effect size: a hard threshold on a measured empirical quantity
turns a finding into a flaky test. The function warns above 5% and says, correctly, that what is
then wrong is this notebook's *narrative* rather than its arithmetic.

The run is on 9 origins (every 7th) × 3 shuffles at full `M`, seeded per group so the result does
not depend on how the work was scheduled across threads.

In [ ]:
dec = decouple_check(asm.fc_store, tru.truth, asm.labels, asm.origins;
                     grid = grid, cfg = cfg)

dsum = combine(groupby(dec.results, [:margin, :model]),
               :rel_es => (x -> 100median(filter(isfinite, collect(skipmissing(x))))) => :pct_es,
               :rel_vs => (x -> 100median(filter(isfinite, collect(skipmissing(x))))) => :pct_vs)
dcor = combine(groupby(dec.corrs, [:margin, :model]), :mean_abs_cor => mean => :mean_abs_cor,
               :max_abs_cor => maximum => :max_abs_cor)
println("median % change when the pairing is destroyed, and the dependence actually present:")
show(leftjoin(dsum, dcor; on = [:margin, :model]); allrows = true, allcols = true, truncate = 34)
println()

# Does destroying the dependence change the CONCLUSION? That is the sentence worth printing.
for mg in unique(dec.results.margin)
    s = @subset(dec.results, :margin .== mg)
    r0 = combine(groupby(s, :model), :es      => median => :v)
    r1 = combine(groupby(s, :model), :es_shuf => median => :v)
    o0 = sort(r0, :v).model; o1 = sort(r1, :v).model
    @printf("margin %-9s model ranking under ES %s under the decoupled ES\n", mg,
            o0 == o1 ? "is IDENTICAL to" : "DIFFERS from")
end

save_show(plot_decoupling(dec, asm.labels), "../res/15j_decoupling.png");

## §9 — Outputs

Everything 15j writes goes to `res/`, and nowhere else. That is 14j's invariant, and here it is also
part of the no-refit guarantee: nothing in this notebook can touch `dt_intermediate`.

Eleven PNGs and nine CSVs. Every figure's numbers reach a CSV: a PDF cannot be read numerically, so
a panel whose series exist nowhere else is a dead end.

In [ ]:
report_mvscores(mv, joined, summ, conc, shares, dec; summ_h = summ_h, diffs = diffs)

println("\nfiles written:")
for f in sort(filter(f -> startswith(f, "15j_"), readdir("../res")))
    @printf("  %-40s %8.1f KB\n", f, filesize(joinpath("../res", f)) / 1024)
end

# The two views dropped on 2026-10-16 must not survive as stale artefacts: nothing in `res/` records
# its own provenance, so a leftover PNG of a view this notebook no longer stands behind is worse
# than one that was never written.
for stale in ("15j_rel_score_by_age.png", "15j_rel_score_joint.png")
    p = joinpath("../res", stale)
    isfile(p) && @warn "15j: `$(stale)` is a STALE artefact of a dropped view (ES over time / " *
                       "over age and time) — delete it; this notebook no longer writes it."
end